# Lab 11: LoRA Fine-tuning

**Module 11** | Read `notes/11-lora-peft.pdf` before running this notebook.

Attach LoRA adapters to a small GPT-2 model and fine-tune on local instruction JSONL with a fraction of trainable weights.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## LoRA fine-tuning on GPT-2

Low-Rank Adaptation (LoRA) injects small trainable matrices into selected layers of a frozen base model. Only the adapter weights update during fine-tuning, which dramatically cuts memory use and training time compared to full fine-tuning.

In this lab you attach LoRA adapters to GPT-2 (`r=8`) and fine-tune on 500 instruction/response pairs from a local JSONL file. You will inspect how many parameters are actually trainable, then run a short causal-language-modeling loop.


### Load instruction data

Each JSONL row has `instruction`, optional `input`, and `output` fields. We format them as a single text sequence so GPT-2 learns to continue with the response after a fixed prompt prefix.


In [ ]:
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path("..").resolve()
DATA_PATH = ROOT / "datasets" / "instruction_sample.jsonl"

records: list[dict] = []
with DATA_PATH.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 500:
            break
        records.append(json.loads(line))

print(f"Loaded {len(records)} instruction examples from {DATA_PATH.name}")
print("Sample keys:", records[0].keys())


### Attach LoRA adapters

`LoraConfig(r=8)` sets the rank of the low-rank decomposition. For GPT-2 we target the fused query-key-value projection (`c_attn`). The PEFT wrapper freezes the base weights and exposes only the adapter matrices for optimization.


In [ ]:
from peft import LoraConfig, get_peft_model

MODEL_NAME = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.to(device)


### Short training loop

We tokenize each formatted example, pad batches to a fixed length, and minimize next-token cross-entropy. Two epochs over 500 rows is enough to demonstrate the workflow without long GPU time.


In [ ]:
def format_example(rec: dict) -> str:
    prompt = rec["instruction"]
    if rec.get("input"):
        prompt += "\n" + rec["input"]
    return (
        f"### Instruction:\n{prompt}\n### Response:\n{rec['output']}"
        f"{tokenizer.eos_token}"
    )


class InstructionDataset(Dataset):
    def __init__(self, rows: list[dict], max_length: int = 128):
        self.texts = [format_example(r) for r in rows]
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


def collate(batch: list[dict]) -> dict:
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }


dataset = InstructionDataset(records)
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

model.train()
losses: list[float] = []
for epoch in range(2):
    epoch_loss = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        loss = out.loss
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    print(f"Epoch {epoch + 1}/2, avg loss: {avg:.4f}")

print("Final batch loss:", f"{loss.item():.4f}")


### Generate a completion

After fine-tuning, sample a greedy continuation from a held-out prompt to confirm the adapter changed model behavior.


In [ ]:
model.eval()
test_prompt = "### Instruction:\nExplain reinforcement learning\n### Response:\n"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
